## 1 · Install & Imports

In [ ]:
# ============================================================
# CELL 1: ENVIRONMENT AND CONFIGURATION
# ============================================================
import os, random, pickle, json, re, html
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from collections import Counter
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import (
    f1_score, accuracy_score, classification_report,
    matthews_corrcoef, cohen_kappa_score, confusion_matrix, roc_auc_score
)
from google.colab import drive
drive.mount('/content/drive')
!pip install -q torchvision transformers fvcore

BASE = '/content/drive/MyDrive/SeaBERT_Final'
IMG_ROOT = BASE
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'

NUM_CLASSES = 2
NUM_MEMBERS = 5
IMAGE_DIM = 1280
TEXT_DIM = 312
HIDDEN_DIM = 256
EMBED_DIM = 64
NUM_HEADS = 4
DROPOUT = 0.1
BATCH_SIZE = 32
MEMBER_EPOCHS = 100
ENSEMBLE_EPOCHS = 100
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MAX_TEXT_LENGTH = 64
SEED = 42

for folder in ['data/embeddings', 'checkpoints', 'plots', 'results']:
    os.makedirs(os.path.join(BASE, folder), exist_ok=True)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
print("Device:", DEVICE, "| Members:", NUM_MEMBERS, "| Classes:", NUM_CLASSES)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda | Members: 5 | Classes: 2


In [ ]:
# ============================================================
# CELL 2: TEXT CLEANING AND DATASET LOADING
# ============================================================

def clean_text(text):
    text = str(text)
    text = html.unescape(text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'([!?.,])\1+', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_dataframe(df, text_column='description'):
    df = df.copy()
    before = len(df)
    df[text_column] = df[text_column].astype(str).apply(clean_text)
    valid = df[text_column].str.strip().str.len() > 0
    df = df[valid].reset_index(drop=True)
    print(f"Rows: {before} -> {len(df)}")
    return df

df_train = pd.read_csv(f'{BASE}/data/processed/inform_train.csv')
df_dev = pd.read_csv(f'{BASE}/data/processed/inform_dev.csv')
df_test = pd.read_csv(f'{BASE}/data/processed/inform_test.csv')

df_train = clean_dataframe(df_train)
df_dev = clean_dataframe(df_dev)
df_test = clean_dataframe(df_test)

print("\nTrain distribution:", Counter(df_train['text_inform']))
print("Dev distribution:", Counter(df_dev['text_inform']))
print("Test distribution:", Counter(df_test['text_inform']))

Rows: 13608 -> 13608
Rows: 2237 -> 2237
Rows: 2237 -> 2237

Train distribution: Counter({1: 9638, 0: 3970})
Dev distribution: Counter({1: 1612, 0: 625})
Test distribution: Counter({1: 1612, 0: 625})


In [ ]:
# ============================================================
# CELL 3: FROZEN FEATURE EXTRACTION
# ============================================================
import torchvision.transforms as T
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from transformers import AutoTokenizer, AutoModel

TEXT_MODEL_NAME = "huawei-noah/TinyBERT_General_4L_312D"
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
text_model = AutoModel.from_pretrained(TEXT_MODEL_NAME).to(DEVICE).eval()
for p in text_model.parameters(): p.requires_grad = False

image_model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1).features.to(DEVICE).eval()
for p in image_model.parameters(): p.requires_grad = False

image_transform = T.Compose([
    T.Resize((224, 224)), T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def extract_event(filename):
    parts = str(filename).split('/')
    return parts[1] if len(parts) >= 2 else 'unknown'

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# ============================================================
# CELL 4: FEATURE EXTRACTION AND CACHING (faster)
# ============================================================

class RawImageTextDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self): return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        image = Image.open(os.path.join(IMG_ROOT, row['filename'])).convert('RGB')
        image = image_transform(image)
        encoded = tokenizer(str(row['description']), truncation=True, padding='max_length',
                             max_length=MAX_TEXT_LENGTH, return_tensors='pt')
        input_ids = encoded['input_ids'].squeeze(0)
        attention_mask = encoded['attention_mask'].squeeze(0)
        label = int(row['text_inform'])
        event = extract_event(row['filename'])
        return image, input_ids, attention_mask, label, event

def extract_and_cache(dataframe, cache_path, batch_size=128, num_workers=2):
    if os.path.exists(cache_path):
        print("Loading:", cache_path)
        with open(cache_path, 'rb') as file:
            return pickle.load(file)

    dataset = RawImageTextDataset(dataframe)
    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
        persistent_workers=(num_workers > 0),
        prefetch_factor=4 if num_workers > 0 else None,
    )

    image_features, text_features, text_masks, labels, events = [], [], [], [], []
    with torch.no_grad():
        for images, input_ids, attn_masks, batch_labels, batch_events in tqdm(loader, desc=os.path.basename(cache_path)):
            images = images.to(DEVICE, non_blocking=True)
            input_ids = input_ids.to(DEVICE, non_blocking=True)
            attn_masks = attn_masks.to(DEVICE, non_blocking=True)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                img_feat = image_model(images)
                txt_out = text_model(input_ids=input_ids, attention_mask=attn_masks).last_hidden_state
            image_features.append(img_feat.cpu().numpy().astype(np.float16))
            text_features.append(txt_out.cpu().numpy().astype(np.float16))
            text_masks.append(attn_masks.cpu().numpy().astype(np.int64))
            labels.extend(batch_labels)
            events.extend(batch_events)

    data = {
        'img_maps': np.concatenate(image_features, axis=0),
        'txt_seqs': np.concatenate(text_features, axis=0),
        'txt_masks': np.concatenate(text_masks, axis=0),
        'labels': np.array(labels, dtype=np.int64),
        'events': events
    }
    with open(cache_path, 'wb') as file:
        pickle.dump(data, file)
    print(f"Saved {cache_path}: {os.path.getsize(cache_path)/1024**2:.2f} MB")
    return data

train_data = extract_and_cache(df_train, f'{BASE}/data/embeddings/inform_train_belt.pkl')
dev_data = extract_and_cache(df_dev, f'{BASE}/data/embeddings/inform_dev_belt.pkl')
test_data = extract_and_cache(df_test, f'{BASE}/data/embeddings/inform_test_belt.pkl')

print("\nFeature shapes:", train_data['img_maps'].shape, train_data['txt_seqs'].shape)

Loading: /content/drive/MyDrive/SeaBERT_Final/data/embeddings/inform_train_belt.pkl
Loading: /content/drive/MyDrive/SeaBERT_Final/data/embeddings/inform_dev_belt.pkl
Loading: /content/drive/MyDrive/SeaBERT_Final/data/embeddings/inform_test_belt.pkl

Feature shapes: (13608, 1280, 7, 7) (13608, 64, 312)


In [ ]:
# ============================================================
# CELL 5: DATASET AND DATALOADERS
# ============================================================

EVENT2IDX = {e: i for i, e in enumerate(sorted(set(train_data['events'] + dev_data['events'] + test_data['events'])))}

class InformDataset(Dataset):
    def __init__(self, data):
        self.images = torch.from_numpy(data['img_maps']).float()
        self.text = torch.from_numpy(data['txt_seqs']).float()
        self.masks = torch.from_numpy(data['txt_masks']).float()
        self.labels = torch.from_numpy(data['labels']).long()
        self.events = torch.tensor([EVENT2IDX[e] for e in data['events']], dtype=torch.long)

    def __len__(self): return len(self.labels)
    def __getitem__(self, index):
        return self.images[index], self.text[index], self.masks[index], self.labels[index], self.events[index]

train_dataset = InformDataset(train_data)
dev_dataset = InformDataset(dev_data)
test_dataset = InformDataset(test_data)

class_counts = np.bincount(train_data['labels'], minlength=NUM_CLASSES).astype(np.float32)
class_weights = 1.0 / np.sqrt(class_counts)
class_weights = class_weights / class_weights.sum() * NUM_CLASSES
CLASS_WEIGHTS = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
print("Class counts:", class_counts, "| Class weights:", CLASS_WEIGHTS.cpu().numpy())

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

Class counts: [3970. 9638.] | Class weights: [1.2181729 0.7818271]


In [ ]:
# ============================================================
# CELL 6: CORAL HEAD, LOSS AND DECODER
# ============================================================

class CoralHead(nn.Module):
    def __init__(self, in_dim, num_classes=NUM_CLASSES):
        super().__init__()
        self.num_thresholds = num_classes - 1
        self.fc = nn.Linear(in_dim, 1, bias=False)
        self.bias0 = nn.Parameter(torch.zeros(1))
        self.deltas = nn.Parameter(torch.ones(max(self.num_thresholds - 1, 0)))

    def forward(self, x):
        base = self.fc(x)
        biases = [self.bias0]
        b = self.bias0
        for i in range(self.num_thresholds - 1):
            b = b - F.softplus(self.deltas[i])
            biases.append(b)
        biases = torch.cat(biases)
        return base + biases.unsqueeze(0)

class WeightedBCEWithLogitsLoss(nn.Module):
    def __init__(self, pos_weight=None):
        super().__init__()
        self.loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def forward(self, logits, targets):
        # BCEWithLogitsLoss expects float targets and the same shape as logits
        return self.loss_fn(logits.squeeze(-1), targets.float())

def decode_coral(logits, threshold=0.5):
    # binary case: single threshold, single logit column
    probabilities = torch.sigmoid(logits[:, 0])
    return (probabilities > threshold).long()

print("CORAL components ready. Thresholds:", NUM_CLASSES - 1)


CORAL components ready. Thresholds: 1


In [ ]:
# ============================================================
# CELL 7: BALANCED BOOTSTRAP SAMPLING
# ============================================================

train_labels = train_data['labels']
class_pools = {c: np.where(train_labels == c)[0] for c in range(NUM_CLASSES)}
samples_per_class = min(len(v) for v in class_pools.values())
print("Available pool sizes:", {c: len(v) for c, v in class_pools.items()})
print("Bootstrap samples per class:", samples_per_class)

def balanced_bootstrap_sample(class_pools, samples_per_class, seed):
    rng = np.random.RandomState(seed)
    indices = []
    for class_id in range(NUM_CLASSES):
        sampled = rng.choice(class_pools[class_id], size=samples_per_class, replace=True)
        indices.extend(sampled.tolist())
    rng.shuffle(indices)
    return indices

member_indices = [balanced_bootstrap_sample(class_pools, samples_per_class, seed=4000 + i) for i in range(NUM_MEMBERS)]
for i, idx in enumerate(member_indices):
    print(f"Member {i+1}:", Counter(train_labels[idx]))

member_loaders = [
    DataLoader(Subset(train_dataset, idx), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    for idx in member_indices
]

Available pool sizes: {0: 3970, 1: 9638}
Bootstrap samples per class: 3970
Member 1: Counter({np.int64(0): 3970, np.int64(1): 3970})
Member 2: Counter({np.int64(1): 3970, np.int64(0): 3970})
Member 3: Counter({np.int64(1): 3970, np.int64(0): 3970})
Member 4: Counter({np.int64(1): 3970, np.int64(0): 3970})
Member 5: Counter({np.int64(0): 3970, np.int64(1): 3970})


In [ ]:
# ============================================================
# CELL 8: BELT ARCHITECTURE
# Bidirectional Ensemble Learning Transformer
# ============================================================

class AdapterPoolText(nn.Module):
    def __init__(self, input_dim=TEXT_DIM, output_dim=HIDDEN_DIM, bottleneck=64):
        super().__init__()
        self.down = nn.Linear(input_dim, bottleneck)
        self.up = nn.Linear(bottleneck, input_dim)
        self.norm = nn.LayerNorm(input_dim)
        self.projection = nn.Linear(input_dim, output_dim)

    def forward(self, text, mask):
        residual = text
        x = F.gelu(self.down(text))
        x = self.up(x)
        x = self.norm(residual + x)
        mask = mask.unsqueeze(-1)
        x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)
        return self.projection(x)


class BELT(nn.Module):
    """One ensemble member: own image/text projections, own bidirectional
    cross-attention, own self-attention refinement, own 64-D embedding.
    Each of the NUM_MEMBERS BELT instances is independent (not shared) and
    trains on its own balanced bootstrap subset."""
    def __init__(self):
        super().__init__()

        self.image_projection = nn.Sequential(
            nn.Linear(IMAGE_DIM, HIDDEN_DIM),
            nn.LayerNorm(HIDDEN_DIM), nn.ReLU()
        )
        self.text_projection = AdapterPoolText()

        self.image_to_text = nn.MultiheadAttention(HIDDEN_DIM,
                                                   NUM_HEADS,
                                                   dropout=DROPOUT,
                                                   batch_first=True)
        self.text_to_image = nn.MultiheadAttention(HIDDEN_DIM,
                                                   NUM_HEADS,
                                                   dropout=DROPOUT,
                                                   batch_first=True)
        self.image_norm = nn.LayerNorm(HIDDEN_DIM)

        self.text_norm = nn.LayerNorm(HIDDEN_DIM)

        self.self_attention = nn.TransformerEncoderLayer(
            d_model=HIDDEN_DIM, nhead=NUM_HEADS, dim_feedforward=512,
            dropout=DROPOUT, batch_first=True, norm_first=True
        )

        self.embedding = nn.Sequential(
            nn.Linear(HIDDEN_DIM * 2, EMBED_DIM),
            nn.LayerNorm(EMBED_DIM), nn.ReLU()
        )
        self.coral_head = CoralHead(EMBED_DIM,
                                    num_classes=NUM_CLASSES)

    def embed(self, image, text, mask):
        if image.ndim == 4:
            image = F.adaptive_avg_pool2d(image, 1).flatten(1)
        v = self.image_projection(image)
        t = self.text_projection(text, mask)

        v_tok, t_tok = v.unsqueeze(1), t.unsqueeze(1)

        v_att, _ = self.image_to_text(v_tok, t_tok, t_tok)   # V <- T
        t_att, _ = self.text_to_image(t_tok, v_tok, v_tok)   # T <- V (parallel, from original tokens)

        v_new = self.image_norm(v_tok + v_att)
        t_new = self.text_norm(t_tok + t_att)

        pair = torch.cat([v_new, t_new], dim=1)          # [B, 2, HIDDEN_DIM]
        pair = self.self_attention(pair)
        fused = pair.flatten(start_dim=1)                # [B, 2*HIDDEN_DIM]

        return self.embedding(fused)                      # [B, EMBED_DIM]

    def forward(self, image, text, mask):
        return self.coral_head(self.embed(image, text, mask))


# Shape check
_m = BELT().to(DEVICE)
_img, _txt, _mask, _label, _event = next(iter(train_loader))
_img, _txt, _mask = _img.to(DEVICE), _txt.to(DEVICE), _mask.to(DEVICE)
with torch.no_grad():
    _emb = _m.embed(_img, _txt, _mask)
    _logits = _m(_img, _txt, _mask)
print("Embedding shape:", tuple(_emb.shape), "| CORAL logits shape:", tuple(_logits.shape))
del _m, _img, _txt, _mask, _label, _event, _emb, _logits

Embedding shape: (32, 64) | CORAL logits shape: (32, 1)


In [ ]:
# ============================================================
# CELL 9: MEMBER EVALUATION
# ============================================================
@torch.no_grad()
def evaluate_member(member, loader):
    member.eval()
    predictions, targets = [], []
    for image, text, mask, label, event in loader:
        image, text, mask = image.to(DEVICE), text.to(DEVICE), mask.to(DEVICE)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = member(image, text, mask)
        predictions.extend(decode_coral(logits).cpu().numpy())
        targets.extend(label.numpy())
    return f1_score(targets, predictions, average='macro', zero_division=0)

In [ ]:
# ============================================================
# CELL 10: TRAIN ONE MEMBER
# ============================================================
def train_member(member_id, epochs=MEMBER_EPOCHS, patience_limit=50):
    set_seed(SEED + member_id)
    member = BELT().to(DEVICE)
    criterion = WeightedBCEWithLogitsLoss(pos_weight=CLASS_WEIGHTS[1])
    optimizer = torch.optim.AdamW(member.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    loader = member_loaders[member_id]

    best_f1, best_state, patience = -np.inf, None, 0
    for epoch in range(epochs):
        member.train()
        running_loss, samples = 0.0, 0
        for image, text, mask, label, event in loader:
            image, text, mask, label = image.to(DEVICE), text.to(DEVICE), mask.to(DEVICE), label.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits = member(image, text, mask)
                loss = criterion(logits, label)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(member.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * label.size(0)
            samples += label.size(0)

        dev_f1 = evaluate_member(member, dev_loader)
        print(f"Member {member_id+1} | Epoch {epoch+1:02d} | Loss {running_loss/samples:.4f} | DEV Macro-F1 {dev_f1:.4f}")
        if dev_f1 > best_f1:
            best_f1, patience = dev_f1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in member.state_dict().items()}
        else:
            patience += 1
        if patience >= patience_limit:
            print("Early stopping."); break

    member.load_state_dict(best_state)
    for p in member.parameters(): p.requires_grad = False
    member.eval()
    print(f"\nMember {member_id+1} best DEV Macro-F1: {best_f1:.4f}")
    return member


In [ ]:
# ============================================================
# CELL 11: TRAIN THE ENSEMBLE MEMBERS
# ============================================================
members = []
for member_id in range(NUM_MEMBERS):
    print("\n" + "="*60)
    print(f"TRAINING MEMBER {member_id+1}/{NUM_MEMBERS}")
    print("="*60)
    members.append(train_member(member_id))
print("\nAll members trained.")


TRAINING MEMBER 1/5
Member 1 | Epoch 01 | Loss 0.4559 | DEV Macro-F1 0.6949
Member 1 | Epoch 02 | Loss 0.3966 | DEV Macro-F1 0.7238
Member 1 | Epoch 03 | Loss 0.3678 | DEV Macro-F1 0.7236
Member 1 | Epoch 04 | Loss 0.3243 | DEV Macro-F1 0.7410
Member 1 | Epoch 05 | Loss 0.2840 | DEV Macro-F1 0.7006
Member 1 | Epoch 06 | Loss 0.2503 | DEV Macro-F1 0.7368
Member 1 | Epoch 07 | Loss 0.2096 | DEV Macro-F1 0.7151
Member 1 | Epoch 08 | Loss 0.1778 | DEV Macro-F1 0.7199
Member 1 | Epoch 09 | Loss 0.1622 | DEV Macro-F1 0.7147
Member 1 | Epoch 10 | Loss 0.1354 | DEV Macro-F1 0.7092
Member 1 | Epoch 11 | Loss 0.1152 | DEV Macro-F1 0.7035
Member 1 | Epoch 12 | Loss 0.0975 | DEV Macro-F1 0.7160
Member 1 | Epoch 13 | Loss 0.0920 | DEV Macro-F1 0.7324
Member 1 | Epoch 14 | Loss 0.0717 | DEV Macro-F1 0.7077
Member 1 | Epoch 15 | Loss 0.0593 | DEV Macro-F1 0.7024
Member 1 | Epoch 16 | Loss 0.0574 | DEV Macro-F1 0.7127
Member 1 | Epoch 17 | Loss 0.0592 | DEV Macro-F1 0.7222
Member 1 | Epoch 18 | Loss 

In [ ]:
# ============================================================
# CELL 12: EMBEDDING-LEVEL ENSEMBLE
# ============================================================
class EmbeddingEnsemble(nn.Module):
    def __init__(self, members):
        super().__init__()
        self.members = nn.ModuleList(members)
        self.num_members = len(members)
        self.weight_logits = nn.Parameter(torch.zeros(self.num_members))
        self.head = CoralHead(EMBED_DIM, num_classes=NUM_CLASSES)

    def get_weights(self):
        return torch.softmax(self.weight_logits, dim=0)

    def get_member_embeddings(self, image, text, mask):
        embeddings = []
        with torch.no_grad():
            for member in self.members:
                embeddings.append(member.embed(image, text, mask))
        return torch.stack(embeddings, dim=0)

    def forward(self, image, text, mask):
        embeddings = self.get_member_embeddings(image, text, mask)
        weights = self.get_weights().view(-1, 1, 1)
        ensemble_embedding = (embeddings * weights).sum(dim=0)
        return self.head(ensemble_embedding)


@torch.no_grad()
def collect_logits(model, loader):
    model.eval()
    all_logits, all_targets = [], []
    for image, text, mask, label, event in loader:
        image, text, mask = image.to(DEVICE), text.to(DEVICE), mask.to(DEVICE)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = model(image, text, mask)
        all_logits.append(logits.float().cpu())
        all_targets.append(label.cpu())
    return torch.cat(all_logits, dim=0), torch.cat(all_targets, dim=0)


def evaluate_model(model, loader, threshold=0.5):
    logits, targets = collect_logits(model, loader)
    preds = decode_coral(logits, threshold=threshold)
    targets_np, preds_np = targets.numpy(), preds.numpy()
    return {
        'macro_f1': f1_score(targets_np, preds_np, average='macro', zero_division=0),
        'weighted_f1': f1_score(targets_np, preds_np, average='weighted', zero_division=0),
        'accuracy': accuracy_score(targets_np, preds_np),
        'mcc': matthews_corrcoef(targets_np, preds_np),
        'kappa': cohen_kappa_score(targets_np, preds_np),
        'auc': roc_auc_score(targets_np, torch.sigmoid(logits[:, 0]).numpy()),
        'targets': targets_np, 'preds': preds_np,
    }

In [ ]:
# ============================================================
# CELL 13: TRAIN THE EMBEDDING ENSEMBLE
# ============================================================
ensemble = EmbeddingEnsemble(members).to(DEVICE)
criterion = WeightedBCEWithLogitsLoss(pos_weight=CLASS_WEIGHTS[1])
optimizer = torch.optim.AdamW([ensemble.weight_logits, *ensemble.head.parameters()],
                               lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

best_f1, best_state, patience, PATIENCE = -np.inf, None, 0, 50

for epoch in range(ENSEMBLE_EPOCHS):
    ensemble.train()
    running_loss, total_samples = 0.0, 0
    for image, text, mask, label, event in train_loader:
        image, text, mask, label = image.to(DEVICE), text.to(DEVICE), mask.to(DEVICE), label.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = ensemble(image, text, mask)
            loss = criterion(logits, label)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([ensemble.weight_logits, *ensemble.head.parameters()], 1.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * label.size(0)
        total_samples += label.size(0)

    dev_result = evaluate_model(ensemble, dev_loader)
    weights = ensemble.get_weights().detach().cpu().numpy()
    print(f"Epoch {epoch+1:03d} | Loss {running_loss/total_samples:.4f} | "
          f"DEV Macro-F1 {dev_result['macro_f1']:.4f} | DEV Acc {dev_result['accuracy']:.4f} | "
          f"Weights {np.round(weights, 4)}")

    if dev_result['macro_f1'] > best_f1:
        best_f1, patience = dev_result['macro_f1'], 0
        best_state = {k: v.detach().cpu().clone() for k, v in ensemble.state_dict().items()}
        print("  ✓ Best model updated.")
    else:
        patience += 1
    if patience >= PATIENCE:
        print("\nEarly stopping."); break

ensemble.load_state_dict(best_state)
ensemble.eval()
print("\nBest DEV Macro-F1:", f"{best_f1:.4f}")
print("Final ensemble weights:", np.round(ensemble.get_weights().detach().cpu().numpy(), 4))


Epoch 001 | Loss 0.3788 | DEV Macro-F1 0.7636 | DEV Acc 0.8145 | Weights [0.1519 0.232  0.2132 0.2396 0.1632]
  ✓ Best model updated.
Epoch 002 | Loss 0.2992 | DEV Macro-F1 0.7605 | DEV Acc 0.8114 | Weights [0.1454 0.2453 0.2643 0.1958 0.1492]
Epoch 003 | Loss 0.2907 | DEV Macro-F1 0.7600 | DEV Acc 0.8114 | Weights [0.1536 0.2508 0.2848 0.1669 0.1438]
Epoch 004 | Loss 0.2888 | DEV Macro-F1 0.7593 | DEV Acc 0.8091 | Weights [0.1669 0.2514 0.2908 0.1495 0.1414]
Epoch 005 | Loss 0.2865 | DEV Macro-F1 0.7593 | DEV Acc 0.8109 | Weights [0.1808 0.2501 0.2924 0.1363 0.1404]
Epoch 006 | Loss 0.2850 | DEV Macro-F1 0.7605 | DEV Acc 0.8109 | Weights [0.1933 0.2472 0.295  0.1263 0.1383]
Epoch 007 | Loss 0.2849 | DEV Macro-F1 0.7593 | DEV Acc 0.8105 | Weights [0.2032 0.2458 0.2942 0.1193 0.1375]
Epoch 008 | Loss 0.2844 | DEV Macro-F1 0.7608 | DEV Acc 0.8109 | Weights [0.211  0.2422 0.2978 0.1126 0.1363]
Epoch 009 | Loss 0.2832 | DEV Macro-F1 0.7602 | DEV Acc 0.8118 | Weights [0.2166 0.2409 0.2964 0

In [ ]:
# ============================================================
# CELL 14: OPTIMIZE CORAL THRESHOLD
# ============================================================
dev_logits, dev_targets = collect_logits(ensemble, dev_loader)
dev_targets_np = dev_targets.numpy()

best_result = None
for t in np.arange(0.20, 0.81, 0.01):
    preds = decode_coral(dev_logits, threshold=t).numpy()
    macro_f1 = f1_score(dev_targets_np, preds, average='macro', zero_division=0)
    acc = accuracy_score(dev_targets_np, preds)
    score = macro_f1 + 0.20 * acc
    if best_result is None or score > best_result['score']:
        best_result = {'t': float(t), 'macro_f1': float(macro_f1), 'accuracy': float(acc), 'score': float(score)}

BEST_T = best_result['t']
print(f"Best DEV threshold: t={BEST_T:.3f} | DEV Macro-F1={best_result['macro_f1']:.4f} | DEV Acc={best_result['accuracy']:.4f}")

Best DEV threshold: t=0.570 | DEV Macro-F1=0.7651 | DEV Acc=0.8096


In [ ]:
# ============================================================
# CELL 15: FINAL TEST EVALUATION
# ============================================================
test_result = evaluate_model(ensemble, test_loader, threshold=BEST_T)
targets, predictions = test_result['targets'], test_result['preds']

print("\n" + "="*60)
print("FINAL TEST RESULTS")
print("="*60)
print(f"Macro F1        : {test_result['macro_f1']:.4f}")
print(f"Weighted F1     : {test_result['weighted_f1']:.4f}")
print(f"Accuracy        : {test_result['accuracy']:.4f}")
print(f"MCC             : {test_result['mcc']:.4f}")
print(f"Cohen Kappa     : {test_result['kappa']:.4f}")
print(f"AUC-ROC         : {test_result['auc']:.4f}")
print(f"\nThreshold: t={BEST_T:.3f}")
print(f"Ensemble weights: {np.round(ensemble.get_weights().detach().cpu().numpy(), 4)}")
print("\nClassification Report:")
print(classification_report(targets, predictions, target_names=["Not Informative", "Informative"], digits=4, zero_division=0))
print("\nConfusion Matrix:")
print(confusion_matrix(targets, predictions))

results = {
    'macro_f1': float(test_result['macro_f1']), 'weighted_f1': float(test_result['weighted_f1']),
    'accuracy': float(test_result['accuracy']), 'mcc': float(test_result['mcc']),
    'kappa': float(test_result['kappa']), 'auc': float(test_result['auc']),
    'threshold': BEST_T, 'ensemble_weights': ensemble.get_weights().detach().cpu().numpy().tolist(),
    'num_members': NUM_MEMBERS, 'embedding_dimension': EMBED_DIM, 'bootstrap_size_per_class': samples_per_class
}
with open(f'{BASE}/results/belt_inform_results.json', 'w') as f:
    json.dump(results, f, indent=2)
torch.save(ensemble.state_dict(), f'{BASE}/checkpoints/belt_inform.pth')
print(f"\nSaved results and model.")


FINAL TEST RESULTS
Macro F1        : 0.7647
Weighted F1     : 0.8094
Accuracy        : 0.8082
MCC             : 0.5297
Cohen Kappa     : 0.5295
AUC-ROC         : 0.8595

Threshold: t=0.570
Ensemble weights: [0.1519 0.232  0.2132 0.2396 0.1632]

Classification Report:
                 precision    recall  f1-score   support

Not Informative     0.6508    0.6768    0.6635       625
    Informative     0.8727    0.8592    0.8659      1612

       accuracy                         0.8082      2237
      macro avg     0.7617    0.7680    0.7647      2237
   weighted avg     0.8107    0.8082    0.8094      2237


Confusion Matrix:
[[ 423  202]
 [ 227 1385]]

Saved results and model.
